# 🧬 Phân tích Dữ liệu PROVEDIt (Exploratory Data Analysis)

Notebook này giúp kiểm tra và trực quan hóa phân bổ của dữ liệu DNA theo:
1. **NOC** (Number of Contributors - Số người đóng góp)
2. **Multiplex** (Bộ kit)
3. **Injection Time** (Thời gian tiêm mẫu)
4. **Sự kết hợp (Strata)** giữa các biến này
5. **Trước và Sau khi Downsample**

In [3]:
!pip3 install pandas

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 3.9 MB/s eta 0:00:01
     |████████████████████████████████| 348 kB 4.5 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 21.8 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [4]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cài đặt giao diện biểu đồ
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

DATA_DIR = "PROVEDIt_1-5-Person CSVs Filtered"

ModuleNotFoundError: No module named 'seaborn'

## 1. Phân tích Dữ liệu Gốc (Dựa trên cấu trúc thư mục)
Việc quét thư mục sẽ cực kỳ nhanh so với đọc hàng trăm ngàn dòng CSV. Ta có thể lấy được NOC, Mux, và Injection Time từ tên thư mục.

In [ ]:
records = []

# Quét tất cả các file CSV trong thư mục gốc
for fpath in glob.glob(f"{DATA_DIR}/**/*.csv", recursive=True):
    parts = fpath.split(os.sep)
    if len(parts) >= 4:
        # Ví dụ: PROVEDIt_1-5-Person CSVs Filtered / PROVEDIt_1-5-Person CSVs Filtered_3130_IDPlus28cycles / 1-Person / 10 sec / file.csv
        mux_folder = parts[1]
        noc_folder = parts[2]
        inj_folder = parts[3]
        
        # Parse Multiplex
        mux = "Unknown"
        if "IDPlus28" in mux_folder: mux = "IDPlus28"
        elif "IDPlus29" in mux_folder: mux = "IDPlus29"
        elif "GF29" in mux_folder: mux = "GF29"
        elif "PP16HS32" in mux_folder: mux = "PP16HS32"
            
        # Parse NOC
        try:
            noc = int(noc_folder.split('-')[0])
        except:
            noc = -1
            
        # Parse Injection Time
        inj = inj_folder.strip()
        
        records.append({
            "Filename": parts[-1],
            "Multiplex": mux,
            "NOC": noc,
            "Injection_Time": inj
        })

df_meta = pd.DataFrame(records)
print(f"🔥 Đã quét xong {len(df_meta):,} file CSV gốc.")
display(df_meta.head())

### 1.1 Phân bổ theo số lượng người đóng góp (NOC)

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_meta, x='NOC', palette='viridis')
plt.title('Phân bổ File CSV theo NOC (Dữ liệu gốc)')
plt.xlabel('Number of Contributors (NOC)')
plt.ylabel('Số lượng file')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.show()

*(Lưu ý: Dữ liệu NOC=1 chênh lệch cực kỳ lớn so với các NOC khác, đây là lý do tại sao ta phải có bước Dynamic Balance Cap 5x)*

### 1.2 Phân bổ theo Bộ Kit (Multiplex)

In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df_meta, x='Multiplex', order=df_meta['Multiplex'].value_counts().index, palette='magma')
plt.title('Phân bổ File theo Bộ Kit (Multiplex)')
plt.ylabel('Số lượng file')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.show()

### 1.3 Phân bổ theo Thời gian Injection

In [ ]:
plt.figure(figsize=(10, 5))
ax = sns.countplot(data=df_meta, x='Injection_Time', order=df_meta['Injection_Time'].value_counts().index, palette='coolwarm')
plt.title('Phân bổ theo Injection Time')
plt.ylabel('Số lượng file')
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.show()

### 1.4 Heatmap tương quan Multiplex vs Injection Time

In [ ]:
cross_tab = pd.crosstab(df_meta['Multiplex'], df_meta['Injection_Time'])
plt.figure(figsize=(10, 6))
sns.heatmap(cross_tab, annot=True, fmt='d', cmap='YlGnBu')
plt.title('Số lượng file theo Kit và Injection Time')
plt.show()

## 2. Phân tích Dữ liệu Đã Tiền xử lý (four_union)
Kiểm tra file `four_union_processed.csv` để xem sau khi preprocessing (gộp tệp, lọc marker), dữ liệu trông như thế nào. File này không bị áp dụng drop class 1 nên nó chứa toàn bộ data hợp lệ.

In [ ]:
processed_file = "data/processed/four_union_processed.csv"
if os.path.exists(processed_file):
    print(f"Đang load {processed_file}...")
    df_proc = pd.read_csv(processed_file)
    
    # Tính số profile thực tế
    n_profiles = df_proc['Sample File'].nunique()
    print(f"✅ Tệp processed có: {n_profiles:,} profiles, {len(df_proc):,} dòng.")
    
    # Gom nhóm theo profile để xem metadata
    profile_info = df_proc.groupby('Sample File').agg(
        NOC=('NOC', 'first'),
        multiplex=('multiplex', 'first'),
        injection_time=('injection_time', 'first'),
        n_padded_loci=('Missing_Marker', 'sum')
    ).reset_index()
else:
    print("File processed chưa tồn tại. Hãy chạy train_xgb.py để tạo.")

### 2.1 Số lượng Marker bị Pad (Missing_Marker) theo từng Kit

In [ ]:
if 'profile_info' in locals():
    # Decode multiplex
    mux_map = {0: 'IDPlus28', 1: 'IDPlus29', 2: 'GF29', 3: 'PP16HS32'}
    profile_info['mux_name'] = profile_info['multiplex'].map(mux_map)
    
    plt.figure(figsize=(8, 5))
    sns.boxplot(data=profile_info, x='mux_name', y='n_padded_loci', palette='Set2')
    plt.title('Số loci bị thiếu (cần Zero-Padding) của mỗi Kit so với mốc Hợp (24 markers)')
    plt.xlabel('Bộ Kit')
    plt.ylabel('Số Locus thiếu')
    plt.show()
    
    display(profile_info.groupby('mux_name')['n_padded_loci'].mean().to_frame('Trung bình số locus bị pad'))

### 2.2 Tỉ lệ Train/Test Split Downsample (Mô phỏng Dataset.py)
Kiểm tra cách hàm `prepare_profile_datasets` chia dữ liệu ra sao.

In [ ]:
if 'profile_info' in locals():
    # Tạo strata như trong dataset.py
    profile_info['strata'] = profile_info['NOC'].astype(str) + '_' + \
                             profile_info['injection_time'].astype(str) + '_' + \
                             profile_info['multiplex'].astype(str)
                             
    print(f"Tổng số nhóm phân lớp (Strata) khác biệt: {profile_info['strata'].nunique()}")
    
    # Tìm cap cho NOC=1
    counts = profile_info['NOC'].value_counts()
    other_counts = [counts.get(n, 0) for n in [2, 3, 4, 5]]
    mean_other = sum(other_counts) / 4.0
    cap_1 = max(1, int(mean_other * 5))
    
    print(f"\n🔥 THÔNG SỐ CÂN BẰNG TỰ ĐỘNG:")
    print(f"- Trung bình mẫu NOC từ 2-5: {mean_other:,.0f} profiles")
    print(f"- Giới hạn (Cap) tối đa cho NOC=1: {cap_1:,} profiles (Gấp 5 lần)")
    print(f"- Số lượng NOC=1 gốc: {counts.get(1, 0):,}")

    # Plot trước và sau giả định
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Trước khi downsample
    sns.barplot(x=counts.index, y=counts.values, ax=axes[0], palette='Reds')
    axes[0].set_title('Trước khi Downsample')
    axes[0].set_xlabel('NOC')
    
    # Sau khi downsample
    counts_after = counts.copy()
    counts_after[1] = min(counts_after[1], cap_1)
    sns.barplot(x=counts_after.index, y=counts_after.values, ax=axes[1], palette='Greens')
    axes[1].set_title('Sau khi Downsample Động (Cap = 5x Mean)')
    axes[1].set_xlabel('NOC')
    
    plt.tight_layout()
    plt.show()